# Trading Engine Monitoring Dashboard
Real-time monitoring: node health, P&L, latency, order book depth.

In [ ]:
%matplotlib widget
import threading
import time
from collections import deque, defaultdict

import numpy as np
import zmq
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt

from engine.config import load_topology
from proto.messages_pb2 import Heartbeat, ExecutionReport, MarketDataTick, KillSwitch

In [ ]:
topo = load_topology()

HEARTBEAT_INTERVAL_S = 0.1   # feed emits 1 HB per 1000 ticks at ~10K tps
STALE_THRESHOLD = 2           # missed HBs before red indicator
KNOWN_NODES = ["feed-0", "matching-engine-0", "risk-gateway-0"]

stop_event = threading.Event()

# Shared state — written by poll thread, read by update functions
last_hb_time = {}             # node_id -> time.monotonic()
last_hb_data = {}             # node_id -> Heartbeat proto

latency_samples = deque(maxlen=600)   # ~60s rolling window at up to 10/s
pnl_per_strategy = defaultdict(float) # strategy_id -> cumulative fill notional
pnl_timestamps = defaultdict(list)    # strategy_id -> list of time.monotonic()
pnl_values = defaultdict(list)        # strategy_id -> list of float

# LOB depth — best bid/ask per symbol from latest ticks
depth_state = {}   # symbol -> {"bid": int, "ask": int, "bid_size": int, "ask_size": int}
active_symbol = "BTC"   # shown in depth chart; selectable via dropdown

In [ ]:
def _make_node_html(node_id, hb=None, stale=True):
    color = "red" if stale else "green"
    status = "STALE" if stale else "OK"
    cpu = f"{hb.cpu_pct:.1f}%" if hb and hb.cpu_pct else "\u2014"
    mem = f"{hb.mem_pct:.1f}%" if hb and hb.mem_pct else "\u2014"
    orders = str(hb.orders_processed) if hb else "\u2014"
    stale_tag = '<br><span style="color:red;font-weight:bold">\u26a0 STALE</span>' if stale else ""
    return (
        f'<div style="border:2px solid {color};border-radius:4px;padding:6px;margin:4px;min-width:160px;">'
        f'<b style="color:{color}">{node_id}</b> <span style="color:{color}">({status})</span><br>'
        f'CPU: {cpu} | MEM: {mem}<br>'
        f'Orders: {orders}'
        f'{stale_tag}'
        f'</div>'
    )

node_displays = {
    nid: widgets.HTML(value=_make_node_html(nid, stale=True))
    for nid in KNOWN_NODES
}

def update_node_health():
    now = time.monotonic()
    for nid, w in node_displays.items():
        hb = last_hb_data.get(nid)
        last_t = last_hb_time.get(nid)
        stale = (last_t is None) or ((now - last_t) > STALE_THRESHOLD * HEARTBEAT_INTERVAL_S)
        w.value = _make_node_html(nid, hb, stale=stale)

In [ ]:
def _poll_loop():
    import zmq as _zmq  # new context — never share with main thread
    ctx = _zmq.Context(io_threads=1)

    hb_sub = ctx.socket(_zmq.SUB)
    hb_sub.setsockopt(_zmq.LINGER, 100)
    hb_sub.setsockopt(_zmq.SUBSCRIBE, b"")
    hb_sub.connect(topo.get_connect_addr("heartbeat_pub"))
    hb_sub.connect(topo.get_connect_addr("me_heartbeat_pub"))
    hb_sub.connect(topo.get_connect_addr("rg_heartbeat_pub"))

    exec_sub = ctx.socket(_zmq.SUB)
    exec_sub.setsockopt(_zmq.LINGER, 100)
    exec_sub.setsockopt(_zmq.RCVHWM, 10000)
    exec_sub.setsockopt(_zmq.SUBSCRIBE, b"")  # all strategy topics
    exec_sub.connect(topo.get_connect_addr("exec_report_pub"))

    feed_sub = ctx.socket(_zmq.SUB)
    feed_sub.setsockopt(_zmq.LINGER, 100)
    feed_sub.setsockopt(_zmq.SUBSCRIBE, b"")
    feed_sub.connect(topo.get_connect_addr("feed_pub"))

    poller = _zmq.Poller()
    poller.register(hb_sub, _zmq.POLLIN)
    poller.register(exec_sub, _zmq.POLLIN)
    poller.register(feed_sub, _zmq.POLLIN)

    while not stop_event.is_set():
        socks = dict(poller.poll(timeout=50))  # 50ms

        if hb_sub in socks:
            raw = hb_sub.recv()
            hb = Heartbeat()
            hb.ParseFromString(raw)
            last_hb_time[hb.node_id] = time.monotonic()
            last_hb_data[hb.node_id] = hb

        if exec_sub in socks:
            # exec_report_pub uses topic-prefix framing: recv_multipart gives [topic, payload]
            # or single-frame if no topic prefix — handle both
            try:
                frames = exec_sub.recv_multipart(flags=_zmq.NOBLOCK)
                raw = frames[-1]  # last frame is always the proto payload
            except _zmq.Again:
                raw = None
            if raw:
                er = ExecutionReport()
                er.ParseFromString(raw)
                from proto.messages_pb2 import ExecType
                if er.exec_type in (ExecType.EXEC_TYPE_FILL, ExecType.EXEC_TYPE_PARTIAL):
                    fill_val = er.fill_price * er.fill_qty / 1e4
                    sid = er.strategy_id or "unknown"
                    t = time.monotonic()
                    pnl_per_strategy[sid] += fill_val
                    pnl_timestamps[sid].append(t)
                    pnl_values[sid].append(pnl_per_strategy[sid])
                    # latency: time between fill_timestamp_ns and now
                    if er.timestamp_ns:
                        latency_us = (time.time_ns() - er.timestamp_ns) / 1e3
                        if 0 < latency_us < 1_000_000:
                            latency_samples.append(latency_us)

        if feed_sub in socks:
            try:
                raw = feed_sub.recv(flags=_zmq.NOBLOCK)
            except _zmq.Again:
                raw = None
            if raw:
                tick = MarketDataTick()
                tick.ParseFromString(raw)
                sym = tick.symbol
                depth_state[sym] = {
                    "bid": tick.bid, "ask": tick.ask,
                    "bid_size": tick.bid_size, "ask_size": tick.ask_size,
                }

    hb_sub.close(); exec_sub.close(); feed_sub.close(); ctx.term()

poll_thread = threading.Thread(target=_poll_loop, daemon=True)
poll_thread.start()
print("Polling thread started.")

In [ ]:
def _schedule_refresh():
    if not stop_event.is_set():
        update_node_health()
        try:
            update_pnl_chart()
        except Exception:
            pass
        try:
            update_latency_hist()
        except Exception:
            pass
        try:
            update_depth_chart()
        except Exception:
            pass
        threading.Timer(1.0, _schedule_refresh).start()

_schedule_refresh()
print("Refresh timer started (1 Hz).")

In [ ]:
health_panel = widgets.HBox(
    list(node_displays.values()),
    layout=widgets.Layout(flex_flow='row wrap')
)
display(health_panel)

In [ ]:
# Run this cell to stop polling cleanly
# stop_event.set()
# print("Polling stopped.")

## P&L Curves (per strategy + aggregate)
Updated each heartbeat interval. P&L = cumulative fill notional (fill_price × fill_qty / 1e4). Aggregate = sum across all strategies. Note: ExecutionReport has no `side` field — this is fill notional, not realized P&L.

In [ ]:
with plt.ioff():
    fig_pnl, ax_pnl = plt.subplots(figsize=(9, 3))
ax_pnl.set_title("Strategy P&L (fill notional)")
ax_pnl.set_xlabel("Time (s)")
ax_pnl.set_ylabel("Cumulative Notional ($)")
ax_pnl.legend(loc="upper left")

# Pre-create lines for known strategy IDs + aggregate
# Lines for dynamic strategy IDs created on first appearance in update_pnl_chart()
_COLORS = plt.cm.tab10.colors
_pnl_lines = {}   # strategy_id -> Line2D
_agg_line, = ax_pnl.plot([], [], color="black", linewidth=2, linestyle="--", label="Aggregate")

def _get_pnl_line(sid):
    if sid not in _pnl_lines:
        color = _COLORS[len(_pnl_lines) % len(_COLORS)]
        line, = ax_pnl.plot([], [], color=color, label=sid)
        _pnl_lines[sid] = line
        ax_pnl.legend(loc="upper left")
    return _pnl_lines[sid]

def update_pnl_chart():
    if not pnl_timestamps:
        return
    t0 = min(v[0] for v in pnl_timestamps.values() if v)
    agg_t, agg_v = [], []
    for sid in list(pnl_timestamps.keys()):
        ts = pnl_timestamps[sid]
        vs = pnl_values[sid]
        if not ts:
            continue
        rel_t = [t - t0 for t in ts]
        _get_pnl_line(sid).set_data(rel_t, vs)
        # aggregate: sample at strategy timestamps
        for i, t in enumerate(rel_t):
            agg_t.append(t)
            agg_v.append(sum(pnl_per_strategy.values()))
    if agg_t:
        sorted_pairs = sorted(zip(agg_t, agg_v))
        _agg_line.set_data([p[0] for p in sorted_pairs], [p[1] for p in sorted_pairs])
    ax_pnl.relim()
    ax_pnl.autoscale_view()
    fig_pnl.canvas.draw_idle()

In [ ]:
display(fig_pnl.canvas)

## Latency Histogram (rolling 60s window)
p50/p95/p99 of tick-to-trade latency (time between ExecutionReport.timestamp_ns and dashboard receipt). Updated every second from a rolling deque of up to 600 samples.

In [ ]:
with plt.ioff():
    fig_lat, ax_lat = plt.subplots(figsize=(5, 3))
ax_lat.set_title("Latency (rolling 60s)")
ax_lat.set_ylabel("Latency (\u03bcs)")

def update_latency_hist():
    if len(latency_samples) < 2:
        return
    data = np.array(latency_samples)
    p50 = float(np.percentile(data, 50))
    p95 = float(np.percentile(data, 95))
    p99 = float(np.percentile(data, 99))
    ax_lat.cla()
    bars = ax_lat.bar(["p50", "p95", "p99"], [p50, p95, p99],
                       color=["#2ecc71", "#e67e22", "#e74c3c"])
    ax_lat.set_ylabel("Latency (\u03bcs)")
    ax_lat.set_title(f"Latency \u2014 rolling {len(latency_samples)} samples")
    for bar, val in zip(bars, [p50, p95, p99]):
        ax_lat.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                    f"{val:.0f}\u03bcs", ha="center", va="bottom", fontsize=9)
    fig_lat.canvas.draw_idle()

display(fig_lat.canvas)

## Order Book Depth (best bid/ask from tick stream)
Cumulative bid/ask step chart from MarketDataTick data. Note: feed currently emits single-level LOB (bid_size=ask_size=1). Chart shows best bid/ask spread. Select symbol with dropdown below.

In [ ]:
with plt.ioff():
    fig_depth, ax_depth = plt.subplots(figsize=(5, 3))
ax_depth.set_title(f"Order Book Depth \u2014 {active_symbol}")
ax_depth.set_xlabel("Price (ticks / 100)")
ax_depth.set_ylabel("Cumulative Size")

symbol_selector = widgets.Dropdown(
    options=["BTC", "ETH", "AAPL", "MSFT", "SPY"],
    value="BTC",
    description="Symbol:",
    layout=widgets.Layout(width="200px")
)

def on_symbol_change(change):
    global active_symbol
    active_symbol = change["new"]
    ax_depth.set_title(f"Order Book Depth \u2014 {active_symbol}")

symbol_selector.observe(on_symbol_change, names="value")

def update_depth_chart():
    sym = active_symbol
    d = depth_state.get(sym)
    ax_depth.cla()
    ax_depth.set_xlabel("Price ($)")
    ax_depth.set_ylabel("Cumulative Size")
    ax_depth.set_title(f"Order Book Depth \u2014 {sym}")
    if d:
        bid_p = d["bid"] / 100.0
        ask_p = d["ask"] / 100.0
        bid_sz = d["bid_size"]
        ask_sz = d["ask_size"]
        # Step chart: bid side (green), ask side (red)
        ax_depth.fill_between([bid_p - 10, bid_p], [0, 0], [bid_sz, bid_sz],
                               step="post", alpha=0.5, color="green", label="Bid")
        ax_depth.fill_between([ask_p, ask_p + 10], [0, 0], [ask_sz, ask_sz],
                               step="pre", alpha=0.5, color="red", label="Ask")
        ax_depth.axvline(x=bid_p, color="green", linestyle="--", alpha=0.5)
        ax_depth.axvline(x=ask_p, color="red", linestyle="--", alpha=0.5)
        ax_depth.legend()
    else:
        ax_depth.text(0.5, 0.5, "Waiting for tick data...", ha="center", va="center",
                      transform=ax_depth.transAxes)
    fig_depth.canvas.draw_idle()

display(widgets.VBox([symbol_selector, fig_depth.canvas]))

In [ ]:
charts_row = widgets.HBox([fig_lat.canvas, fig_depth.canvas],
                           layout=widgets.Layout(justify_content="space-around"))
# Note: figures already displayed individually above; this cell is optional combined view
# Uncomment to use combined layout instead:
# display(charts_row)

## Kill Switch
Two-stage activation: ARM → CONFIRM. Sends KillSwitch protobuf to risk gateway (port 5560). Once activated, all pending orders are cancelled and new submissions are rejected system-wide.
⚠ This action cannot be undone without restarting the risk gateway process.

In [ ]:
# Kill switch PUSH socket — created in main thread (not poll thread)
# Uses a separate ZMQ context for the push socket
import zmq as _zmq_ks
_ks_ctx = _zmq_ks.Context(io_threads=1)
kill_push = _ks_ctx.socket(_zmq_ks.PUSH)
kill_push.setsockopt(_zmq_ks.LINGER, 100)
kill_push.connect(topo.get_connect_addr("kill_switch"))
print(f"Kill switch connected to {topo.get_connect_addr('kill_switch')}")


In [ ]:
arm_btn = widgets.Button(
    description="ARM Kill Switch",
    button_style="warning",
    icon="exclamation-triangle",
    layout=widgets.Layout(width="200px", height="40px")
)
confirm_btn = widgets.Button(
    description="CONFIRM KILL",
    button_style="danger",
    icon="stop",
    layout=widgets.Layout(width="180px", height="40px", display="none")
)
cancel_arm_btn = widgets.Button(
    description="Cancel",
    button_style="",
    layout=widgets.Layout(width="100px", height="40px", display="none")
)
ks_status = widgets.HTML(value='<span style="color:gray">Kill switch: SAFE</span>')

def on_arm(b):
    confirm_btn.layout.display = ""
    cancel_arm_btn.layout.display = ""
    ks_status.value = '<span style="color:orange;font-weight:bold">⚠ ARMED — click CONFIRM KILL to halt all trading</span>'

def on_cancel_arm(b):
    confirm_btn.layout.display = "none"
    cancel_arm_btn.layout.display = "none"
    ks_status.value = '<span style="color:gray">Kill switch: SAFE</span>'

def on_confirm(b):
    ks = KillSwitch()
    ks.schema_version = 1
    ks.trigger_timestamp = time.time_ns()
    ks.reason = "Dashboard operator kill switch"
    ks.operator_id = "dashboard"
    try:
        kill_push.send(ks.SerializeToString(), flags=_zmq_ks.NOBLOCK)
        ks_status.value = (
            '<span style="color:red;font-weight:bold;font-size:1.2em">'
            '🛑 KILL SWITCH ACTIVATED — all trading halted'
            '</span>'
        )
    except Exception as e:
        ks_status.value = f'<span style="color:red">Send failed: {e}</span>'
    finally:
        arm_btn.disabled = True
        confirm_btn.disabled = True
        cancel_arm_btn.disabled = True
        arm_btn.description = "KILLED"
        arm_btn.button_style = "danger"
        confirm_btn.layout.display = "none"
        cancel_arm_btn.layout.display = "none"

arm_btn.on_click(on_arm)
cancel_arm_btn.on_click(on_cancel_arm)
confirm_btn.on_click(on_confirm)

kill_panel = widgets.VBox([
    widgets.HBox([arm_btn, confirm_btn, cancel_arm_btn]),
    ks_status
], layout=widgets.Layout(
    border="2px solid #e74c3c",
    border_radius="6px",
    padding="10px",
    margin="10px 0"
))
display(kill_panel)


In [ ]:
# Consolidated dashboard — uncomment to use instead of individual section displays above
# Requires all figure canvases already created in earlier cells

# full_dashboard = widgets.VBox([
#     widgets.HTML("<h2>Trading Engine Monitor</h2>"),
#     health_panel,
#     fig_pnl.canvas,
#     widgets.HBox([fig_lat.canvas, widgets.VBox([symbol_selector, fig_depth.canvas])]),
#     kill_panel,
# ])
# display(full_dashboard)
print("Dashboard ready. Run pipeline processes, then observe panels above.")
print("To stop polling: run stop_event.set() in a new cell.")
